# 🧠 Agent Memory Stores (Preview) 🏦

Welcome to the **Memory Stores** tutorial! A great banking advisor remembers you — your risk appetite, your goals, your last conversation. Foundry **Memory Stores** give an agent durable, per-customer memory that persists across sessions.

In this notebook we'll remember a customer's **investment risk profile** so a wealth advisor can recall it days later:

1. **Initialize** an `AIProjectClient` with the **preview** surface enabled.
2. **Create a memory store** (user-profile + chat-summary memory, backed by chat + embedding models).
3. **Write a memory** for a customer scope (their risk profile).
4. **Recall it** in a fresh "session" via semantic `search_memories`.
5. **Clean up** the store.

### 🔐 Authentication
Sign in with `az login` (Entra ID only — no keys). `DefaultAzureCredential` resolves your CLI session automatically.

### ⚠️ Important Disclaimer
> Educational and demonstration purposes only — not financial advice. Use fictitious customer data; never store real PII in workshop resources.

> 🧪 **Preview:** Memory stores use the `project.beta.*` surface and require `allow_preview=True`. APIs may change.


In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    MemoryStoreDefaultDefinition,
    MemoryStoreDefaultOptions,
    MemoryItemKind,
)

load_dotenv()

project_endpoint = os.environ["AI_FOUNDRY_PROJECT_ENDPOINT"]
chat_model = os.environ.get("AZURE_AI_MODEL_DEPLOYMENT_NAME", "gpt-5.4")
embedding_model = os.environ.get("EMBEDDING_MODEL_DEPLOYMENT_NAME", "text-embedding-3-large")
print(f"🤖 Chat model: {chat_model} | 🔢 Embeddings: {embedding_model}")

# allow_preview=True unlocks beta.memory_stores
project = AIProjectClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(exclude_managed_identity_credential=True),
    allow_preview=True,
)
print("✅ AIProjectClient ready (preview surface enabled)")

## 1. Create a memory store 🗄️

A `MemoryStoreDefaultDefinition` configures *how* memories are extracted and searched: a **chat model** distills conversations, an **embedding model** powers semantic recall. We enable **user profile** (stable preferences) and **chat summary** (conversation highlights).


In [ ]:
STORE_NAME = "wealth-advisor-memory"

store = project.beta.memory_stores.create(
    name=STORE_NAME,
    description="Per-customer memory for a private wealth advisor.",
    definition=MemoryStoreDefaultDefinition(
        chat_model=chat_model,
        embedding_model=embedding_model,
        options=MemoryStoreDefaultOptions(
            user_profile_enabled=True,
            chat_summary_enabled=True,
        ),
    ),
)
print(f"🗄️ Memory store created: {store.name} (id={store.id})")


## 2. Remember the customer's risk profile 👤

Each customer gets their own **scope** so memories stay isolated. We'll write a `USER_PROFILE` memory capturing this customer's investment stance. In production this is usually extracted automatically from conversations; here we set it explicitly to keep the demo deterministic.


In [ ]:
# Scope isolates memories per customer (use a stable, non-PII identifier)
CUSTOMER_SCOPE = "customer-1042"

memory = project.beta.memory_stores.create_memory(
    STORE_NAME,
    scope=CUSTOMER_SCOPE,
    content=(
        "Conservative investor with low risk tolerance. Prefers capital preservation, "
        "bonds, and high-yield savings over equities. Planning to retire in ~5 years."
    ),
    kind=MemoryItemKind.USER_PROFILE,
)
print(f"👤 Stored memory {memory.memory_id} for {CUSTOMER_SCOPE} (kind={memory.kind})")


## 3. Recall it in a new session 🔎

Days later, a different conversation begins. By searching the same scope, the advisor recalls the customer's risk profile — no need to ask again. `search_memories` takes the new conversation turn(s) and returns the most relevant stored memories.


In [ ]:
results = project.beta.memory_stores.search_memories(
    STORE_NAME,
    scope=CUSTOMER_SCOPE,
    items="What should I recommend for this customer's portfolio?",
)

print(f"🔎 Recalled {len(results.memories)} memory(ies) for {CUSTOMER_SCOPE}:\n")
for hit in results.memories:
    print(f"  • [{hit.memory_item.kind}] {hit.memory_item.content}")

if results.memories:
    print("\n💡 The advisor now knows this is a conservative investor — no need to re-ask.")


## 4. Clean up 🧹

Memory stores persist until deleted, so we remove ours to avoid leaking workshop resources. Wrapped in `try/except` so cleanup never fails the notebook.


In [ ]:
try:
    project.beta.memory_stores.delete(STORE_NAME)
    print(f"🗑️ Deleted memory store '{STORE_NAME}'")
except Exception as e:
    print(f"ℹ️ Memory store cleanup skipped: {type(e).__name__}: {e}")

project.close()
print("✅ Cleanup complete")


## 📚 References

- [Memory in Foundry Agent Service (preview)](https://learn.microsoft.com/azure/foundry/agents/concepts/what-is-memory) — what memory is and its types
- [Create and use memory in Foundry Agent Service](https://learn.microsoft.com/azure/foundry/agents/how-to/memory-usage) — create a memory store and use it
- [azure-ai-projects (Python) reference](https://learn.microsoft.com/python/api/overview/azure/ai-projects-readme?view=azure-python) — `beta.memory_stores` APIs
- [Agents, conversations, and responses](https://learn.microsoft.com/azure/foundry/agents/concepts/runtime-components) — where memory fits in a run
> 💡 Tip: search the official docs live from your agent via the **Microsoft Learn MCP** at `https://learn.microsoft.com/api/mcp`.